In [21]:
%pip install pdfplumber sentence-transformers chromadb tf-keras torch -q

Note: you may need to restart the kernel to use updated packages.


In [22]:
import pdfplumber 
from sentence_transformers import SentenceTransformer
import chromadb
import os

# Phase 1 Configuration
CONFIG = {
    "model_name": "all-MiniLM-L6-v2",
    "chunk_size": 300,
    "collection_name": "pdf_collection",
    "top_k_results": 3
}

# Initialize Model
model = SentenceTransformer(CONFIG["model_name"])
print("✓ Model loaded successfully")

✓ Model loaded successfully


In [23]:
def extract_text(pdf_path):
    """
    Extract text from PDF file.
    
    Args:
        pdf_path: Path to PDF file
        
    Returns:
        Extracted text string, or None if error
    """
    text = ""
    try:
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"PDF file not found: {pdf_path}")
            
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)
            for i, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
            print(f"✓ Extracted text from {total_pages} pages ({len(text)} characters)")
        return text
    except Exception as e:
        print(f"✗ Error extracting PDF: {str(e)}")
        return None

In [24]:
def chunk_text(text, chunk_size=None):
    """
    Split text into chunks of specified size.
    
    Args:
        text: Input text to chunk
        chunk_size: Number of words per chunk (default: from CONFIG)
        
    Returns:
        List of text chunks
    """
    if chunk_size is None:
        chunk_size = CONFIG["chunk_size"]
        
    if not text or not text.strip():
        print("✗ Empty text provided")
        return []
    
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    
    print(f"✓ Created {len(chunks)} chunks (size: {chunk_size} words)")
    return chunks

In [25]:
def create_embeddings(chunks):
    """
    Generate embeddings for text chunks.
    
    Args:
        chunks: List of text chunks
        
    Returns:
        Array of embeddings or None if error
    """
    try:
        if not chunks:
            print("✗ No chunks provided")
            return None
            
        embeddings = model.encode(chunks, show_progress_bar=False)
        print(f"✓ Generated {len(embeddings)} embeddings (dim: {len(embeddings[0])})")
        return embeddings
    except Exception as e:
        print(f"✗ Error creating embeddings: {str(e)}")
        return None

In [26]:
def store_in_chromadb(chunks, embeddings):
    """
    Store chunks and embeddings in ChromaDB.
    
    Args:
        chunks: List of text chunks
        embeddings: Array of embeddings
        
    Returns:
        ChromaDB collection or None if error
    """
    try:
        if chunks is None or embeddings is None:
            print("✗ Chunks or embeddings are None")
            return None
            
        if len(chunks) != len(embeddings):
            print("✗ Chunk and embedding count mismatch")
            return None
        
        client = chromadb.Client()
        
        # Clear existing collection if it exists
        try:
            client.delete_collection(name=CONFIG["collection_name"])
        except:
            pass
        
        collection = client.create_collection(name=CONFIG["collection_name"])

        for i, chunk in enumerate(chunks):
            collection.add(
                documents=[chunk],
                embeddings=[embeddings[i].tolist()] if hasattr(embeddings[i], 'tolist') else [embeddings[i]],
                ids=[str(i)]
            )

        print(f"✓ Stored {len(chunks)} chunks in ChromaDB")
        return collection
    except Exception as e:
        print(f"✗ Error storing in ChromaDB: {str(e)}")
        return None

In [27]:
def search(collection, query, top_k=None):
    """
    Perform semantic search on collection.
    
    Args:
        collection: ChromaDB collection
        query: Search query string
        top_k: Number of results to return (default: from CONFIG)
        
    Returns:
        List of most relevant documents
    """
    try:
        if top_k is None:
            top_k = CONFIG["top_k_results"]
            
        if not query or not query.strip():
            print("✗ Empty query")
            return []
            
        query_embedding = model.encode(query).tolist()

        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )

        return results['documents'][0] if results['documents'] else []
    except Exception as e:
        print(f"✗ Error searching: {str(e)}")
        return []